
# Semantic Attention Master Dataset — A–E with Full Diagnostics

This notebook is designed for the folder structure:

```text
SEMANTIC/
├── A/
├── B/
├── C/
├── D/
├── E/
└── this_notebook.ipynb
```

It does **two things at the same time**:

1. Builds the final participant-level master datasets.
2. Explains **exactly why any CSV fails to load**.

It never silently replaces missing Condition × Image files with zeros.

Expected semantic columns in your CSVs:

```text
g_Headline
g_Product
g_CTA
g_Other
```

They are mapped internally to:

\[
G=[g_H,g_P,g_C,g_O].
\]

Expected time columns:

```text
T_full_ms
T_Headline_ms
T_Product_ms
T_CTA_ms
T_Other_ms
```


In [ ]:

from pathlib import Path
import re
import traceback
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 500)

CONDITIONS = ["A", "B", "C", "D", "E"]
EXPECTED_IMAGE_IDS = list(range(1, 21))
EXPECTED_PARTICIPANTS_PER_CELL = 20

G_SUM_TOL = 1e-6
TIME_SUM_TOL_MS = 1e-6
NEGATIVE_TOL_MS = 1e-9

print("Jupyter current directory:", Path.cwd().resolve())


## 1. Find the SEMANTIC root folder

In [ ]:

def looks_like_semantic_root(path):
    path = Path(path)
    return all((path / c).is_dir() for c in CONDITIONS)

def find_semantic_root():
    cwd = Path.cwd().resolve()

    # Most likely case: notebook is already inside SEMANTIC/
    if looks_like_semantic_root(cwd):
        return cwd

    # Check parents
    for p in cwd.parents:
        if looks_like_semantic_root(p):
            return p

    # Check direct children of cwd and its parents
    candidates = [cwd, *cwd.parents]
    for base in candidates:
        try:
            for child in base.iterdir():
                if child.is_dir() and looks_like_semantic_root(child):
                    return child.resolve()
        except Exception:
            pass

    return None

DATA_ROOT = find_semantic_root()

if DATA_ROOT is None:
    print("ERROR: Could not find a folder containing A/, B/, C/, D/, E/.")
    print()
    print("Set DATA_ROOT manually, e.g.:")
    print(r'DATA_ROOT = Path(r"/home/walidtaib/Desktop/semantic")')
    raise RuntimeError("SEMANTIC root folder not found.")
else:
    print("Detected DATA_ROOT:", DATA_ROOT)

OUTPUT_DIR = DATA_ROOT / "semantic_attention_analysis_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print()
for c in CONDITIONS:
    folder = DATA_ROOT / c
    print(f"{c}: exists={folder.exists()} | csv_files={len(list(folder.glob('*.csv')))}")


## 2. Helpers: image IDs, schema mapping, and diagnostic messages

In [ ]:

def parse_image_id(path):
    m = re.search(r"(\d+)", Path(path).stem)
    return int(m.group(1)) if m else None

G_COLUMN_MAP = {
    "g_H": ["g_H", "g_Headline"],
    "g_P": ["g_P", "g_Product"],
    "g_C": ["g_C", "g_CTA"],
    "g_O": ["g_O", "g_Other"],
}

TIME_COLUMN_MAP = {
    "T_full_ms": ["T_full_ms"],
    "T_H_ms": ["T_H_ms", "T_Headline_ms"],
    "T_P_ms": ["T_P_ms", "T_Product_ms"],
    "T_C_ms": ["T_C_ms", "T_CTA_ms"],
    "T_O_ms": ["T_O_ms", "T_Other_ms"],
}

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def canonicalize_participant_file(df):
    """
    Converts the user's CSV schema to canonical analysis columns while
    preserving all original columns.
    """
    out = df.copy()

    if "participant_id" not in out.columns:
        raise ValueError(
            "Missing required column 'participant_id'. "
            f"Available columns: {list(out.columns)}"
        )

    missing_semantic = []
    semantic_sources = {}

    for canonical, aliases in G_COLUMN_MAP.items():
        src = first_existing(out, aliases)
        if src is None:
            missing_semantic.append(
                f"{canonical} expected one of {aliases}"
            )
        else:
            semantic_sources[canonical] = src
            out[canonical] = pd.to_numeric(out[src], errors="coerce")

    if missing_semantic:
        raise ValueError(
            "Missing semantic proportion columns: "
            + " | ".join(missing_semantic)
            + f" | Available columns: {list(out.columns)}"
        )

    # Time columns are optional for loading, but required for full QC.
    time_sources = {}
    for canonical, aliases in TIME_COLUMN_MAP.items():
        src = first_existing(out, aliases)
        time_sources[canonical] = src
        out[canonical] = (
            pd.to_numeric(out[src], errors="coerce")
            if src is not None else np.nan
        )

    return out, semantic_sources, time_sources

def shannon_entropy_rows(frame, cols=("g_H","g_P","g_C","g_O")):
    p = frame[list(cols)].astype(float).to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = np.where(p > 0, p * np.log2(p), 0.0)
    return -np.sum(terms, axis=1)


## 3. Inventory and coverage matrix

In [ ]:

inventory_rows = []
condition_files = {}

for condition in CONDITIONS:
    folder = DATA_ROOT / condition
    files = sorted(
        folder.glob("*.csv"),
        key=lambda p: (
            parse_image_id(p) is None,
            parse_image_id(p) or 10**9,
            p.name,
        )
    )
    condition_files[condition] = files

    for path in files:
        inventory_rows.append({
            "condition": condition,
            "image_id": parse_image_id(path),
            "filename": path.name,
            "path": str(path.resolve()),
        })

inventory = pd.DataFrame(inventory_rows)

print("Discovered CSV files:")
for c in CONDITIONS:
    print(f"  {c}: {len(condition_files[c])}")

display(inventory.head(50))

present_sets = {
    c: {
        parse_image_id(p)
        for p in condition_files[c]
        if parse_image_id(p) is not None
    }
    for c in CONDITIONS
}

coverage = pd.DataFrame(
    {
        c: [img in present_sets[c] for img in EXPECTED_IMAGE_IDS]
        for c in CONDITIONS
    },
    index=EXPECTED_IMAGE_IDS,
)
coverage.index.name = "image_id"

missing_rows = [
    {"condition": c, "image_id": img}
    for c in CONDITIONS
    for img in EXPECTED_IMAGE_IDS
    if img not in present_sets[c]
]
missing_images = pd.DataFrame(
    missing_rows,
    columns=["condition", "image_id"]
)

print("\nMissing Condition × Image files:")
display(missing_images if len(missing_images) else pd.DataFrame({"status":["None"]}))

print("\nCoverage matrix:")
display(coverage)

coverage.reset_index().to_csv(
    OUTPUT_DIR / "coverage_matrix.csv", index=False
)
missing_images.to_csv(
    OUTPUT_DIR / "missing_images_by_condition.csv", index=False
)


## 4. Pre-load schema audit of every CSV

In [ ]:

schema_audit_rows = []

for condition in CONDITIONS:
    for path in condition_files[condition]:
        image_id = parse_image_id(path)

        row = {
            "condition": condition,
            "image_id": image_id,
            "source_file": path.name,
            "path": str(path.resolve()),
            "readable": False,
            "n_rows": np.nan,
            "n_columns": np.nan,
            "has_participant_id": False,
            "g_H_source": None,
            "g_P_source": None,
            "g_C_source": None,
            "g_O_source": None,
            "T_full_source": None,
            "T_H_source": None,
            "T_P_source": None,
            "T_C_source": None,
            "T_O_source": None,
            "schema_status": "UNKNOWN",
            "schema_error": "",
        }

        try:
            df = pd.read_csv(path)

            row["readable"] = True
            row["n_rows"] = len(df)
            row["n_columns"] = len(df.columns)
            row["has_participant_id"] = "participant_id" in df.columns

            for canonical, aliases in G_COLUMN_MAP.items():
                row[f"{canonical}_source"] = first_existing(df, aliases)

            for canonical, aliases in TIME_COLUMN_MAP.items():
                short = {
                    "T_full_ms":"T_full",
                    "T_H_ms":"T_H",
                    "T_P_ms":"T_P",
                    "T_C_ms":"T_C",
                    "T_O_ms":"T_O",
                }[canonical]
                row[f"{short}_source"] = first_existing(df, aliases)

            missing_required = []

            if "participant_id" not in df.columns:
                missing_required.append("participant_id")

            for canonical, aliases in G_COLUMN_MAP.items():
                if first_existing(df, aliases) is None:
                    missing_required.append(
                        f"{canonical} (one of {aliases})"
                    )

            if missing_required:
                row["schema_status"] = "FAIL"
                row["schema_error"] = (
                    "Missing required columns: "
                    + "; ".join(missing_required)
                    + " | Available columns: "
                    + ", ".join(df.columns.astype(str))
                )
            else:
                row["schema_status"] = "PASS"

        except Exception as e:
            row["schema_status"] = "READ_ERROR"
            row["schema_error"] = (
                f"{type(e).__name__}: {e}"
            )

        schema_audit_rows.append(row)

schema_audit = pd.DataFrame(schema_audit_rows)

print("Schema audit summary:")
display(
    schema_audit.groupby("schema_status")
    .size()
    .reset_index(name="n_files")
)

failed_schema = schema_audit[
    schema_audit["schema_status"] != "PASS"
].copy()

if len(failed_schema):
    print("\nFiles with schema/read problems:")
    display(
        failed_schema[
            [
                "condition","image_id","source_file",
                "schema_status","schema_error"
            ]
        ]
    )
else:
    print("\nAll discovered CSV files passed the schema audit.")

schema_audit.to_csv(
    OUTPUT_DIR / "schema_audit_all_files.csv",
    index=False
)
failed_schema.to_csv(
    OUTPUT_DIR / "schema_audit_failed_files.csv",
    index=False
)


## 5. Load all valid files — with exact rejection reasons

### Loader fix
The source CSVs already contain a `condition` column. This version overwrites metadata columns safely instead of calling `DataFrame.insert()` with duplicate column names.

In [ ]:

master_parts = []
processing_rows = []

for condition in CONDITIONS:
    for path in condition_files[condition]:
        image_id = parse_image_id(path)

        try:
            raw_df = pd.read_csv(path)

            part, semantic_sources, time_sources = (
                canonicalize_participant_file(raw_df)
            )

            # IMPORTANT:
            # Your generated CSVs already contain a "condition" column.
            # Do NOT use DataFrame.insert("condition", ...), because that raises:
            #   ValueError: cannot insert condition, already exists
            #
            # Instead overwrite/set metadata columns safely.
            part["condition"] = condition
            part["image_id"] = image_id
            part["source_file"] = path.name

            # Move the metadata columns to the front without duplicating them.
            front = ["condition", "image_id", "source_file"]
            remaining = [c for c in part.columns if c not in front]
            part = part[front + remaining]

            master_parts.append(part)

            processing_rows.append({
                "condition": condition,
                "image_id": image_id,
                "source_file": path.name,
                "status": "LOADED",
                "rows": len(part),
                "error_type": "",
                "error_message": "",
                "semantic_mapping": str(semantic_sources),
                "time_mapping": str(time_sources),
            })

        except Exception as e:
            processing_rows.append({
                "condition": condition,
                "image_id": image_id,
                "source_file": path.name,
                "status": "NOT LOADED",
                "rows": np.nan,
                "error_type": type(e).__name__,
                "error_message": str(e),
                "semantic_mapping": "",
                "time_mapping": "",
            })

processing_report = pd.DataFrame(processing_rows)

print("Processing summary:")
display(
    processing_report.groupby("status")
    .size()
    .reset_index(name="n_files")
)

print("\nFull processing report:")
display(processing_report)

failed_processing = processing_report[
    processing_report["status"] != "LOADED"
].copy()

if len(failed_processing):
    print("\nRejected files and exact reasons:")
    display(
        failed_processing[
            [
                "condition","image_id","source_file",
                "error_type","error_message"
            ]
        ]
    )

processing_report.to_csv(
    OUTPUT_DIR / "processing_report.csv",
    index=False
)
failed_processing.to_csv(
    OUTPUT_DIR / "processing_failed_files.csv",
    index=False
)

if not master_parts:
    print("\nNO FILES WERE LOADED.")
    print("The exact reasons are printed above and saved to:")
    print(OUTPUT_DIR / "processing_failed_files.csv")
    raise RuntimeError(
        "No valid participant-level CSVs were loaded. "
        "Inspect the diagnostic tables above."
    )

master_all = pd.concat(
    master_parts,
    ignore_index=True,
    sort=False
)

print("\nSUCCESS")
print("Loaded participant rows:", len(master_all))
print(
    "Loaded Condition × Image cells:",
    master_all[["condition","image_id"]]
    .drop_duplicates()
    .shape[0]
)


## 6. Row-level QC with explicit reasons

In [ ]:

G_COLS = ["g_H","g_P","g_C","g_O"]
TIME_COLS = ["T_H_ms","T_P_ms","T_C_ms","T_O_ms"]

master_all["g_sum"] = master_all[G_COLS].sum(axis=1, min_count=4)
master_all["g_sum_error"] = (master_all["g_sum"] - 1.0).abs()

master_all["has_missing_g"] = master_all[G_COLS].isna().any(axis=1)
master_all["has_negative_g"] = (
    master_all[G_COLS] < -G_SUM_TOL
).any(axis=1)
master_all["has_g_above_1"] = (
    master_all[G_COLS] > 1 + G_SUM_TOL
).any(axis=1)
master_all["g_sum_valid"] = (
    master_all["g_sum_error"] <= G_SUM_TOL
)

master_all["has_complete_time"] = master_all[
    ["T_full_ms", *TIME_COLS]
].notna().all(axis=1)

master_all["time_component_sum_ms"] = master_all[
    TIME_COLS
].sum(axis=1, min_count=4)

master_all["time_sum_error_ms"] = (
    master_all["time_component_sum_ms"]
    - master_all["T_full_ms"]
).abs()

master_all["time_sum_valid"] = (
    ~master_all["has_complete_time"]
    | (master_all["time_sum_error_ms"] <= TIME_SUM_TOL_MS)
)

master_all["has_negative_time"] = (
    master_all[TIME_COLS] < -NEGATIVE_TOL_MS
).any(axis=1)

master_all["zero_or_negative_full_time"] = (
    master_all["T_full_ms"].notna()
    & (master_all["T_full_ms"] <= 0)
)

KEY = ["condition","image_id","participant_id"]
master_all["duplicate_condition_image_participant"] = (
    master_all.duplicated(KEY, keep=False)
)

master_all["semantic_attention"] = master_all[
    ["g_H","g_P","g_C"]
].sum(axis=1, min_count=3)

master_all["semantic_attention_from_other"] = (
    1.0 - master_all["g_O"]
)

master_all["semantic_attention_identity_error"] = (
    master_all["semantic_attention"]
    - master_all["semantic_attention_from_other"]
).abs()

master_all["semantic_entropy_bits"] = shannon_entropy_rows(master_all)
master_all["semantic_entropy_normalized"] = (
    master_all["semantic_entropy_bits"] / np.log2(4)
)

def row_error_reason(row):
    reasons = []

    if row["has_missing_g"]:
        reasons.append("missing semantic proportion")
    if row["has_negative_g"]:
        reasons.append("negative semantic proportion")
    if row["has_g_above_1"]:
        reasons.append("semantic proportion > 1")
    if (not row["has_missing_g"]) and (not row["g_sum_valid"]):
        reasons.append(
            f"g_sum != 1 (error={row['g_sum_error']:.6g})"
        )
    if row["has_negative_time"]:
        reasons.append("negative semantic time")
    if row["zero_or_negative_full_time"]:
        reasons.append("T_full_ms <= 0")
    if not row["time_sum_valid"]:
        reasons.append(
            f"time components != T_full_ms "
            f"(error={row['time_sum_error_ms']:.6g} ms)"
        )
    if row["duplicate_condition_image_participant"]:
        reasons.append("duplicate condition-image-participant")

    return "; ".join(reasons)

master_all["qc_error_reason"] = master_all.apply(
    row_error_reason, axis=1
)
master_all["row_qc_pass"] = (
    master_all["qc_error_reason"] == ""
)

quality_summary = pd.DataFrame({
    "check": [
        "total_rows",
        "missing_g",
        "negative_g",
        "g_above_1",
        "invalid_g_sum",
        "negative_time",
        "zero_or_negative_full_time",
        "invalid_time_sum",
        "duplicate_condition_image_participant",
        "qc_pass_rows",
        "qc_fail_rows",
    ],
    "count": [
        len(master_all),
        int(master_all["has_missing_g"].sum()),
        int(master_all["has_negative_g"].sum()),
        int(master_all["has_g_above_1"].sum()),
        int((~master_all["g_sum_valid"] & ~master_all["has_missing_g"]).sum()),
        int(master_all["has_negative_time"].sum()),
        int(master_all["zero_or_negative_full_time"].sum()),
        int((~master_all["time_sum_valid"]).sum()),
        int(master_all["duplicate_condition_image_participant"].sum()),
        int(master_all["row_qc_pass"].sum()),
        int((~master_all["row_qc_pass"]).sum()),
    ]
})

display(quality_summary)

problem_rows = master_all[
    ~master_all["row_qc_pass"]
].copy()

if len(problem_rows):
    cols = [
        "condition","image_id","participant_id","source_file",
        "g_H","g_P","g_C","g_O","g_sum",
        "T_full_ms","T_H_ms","T_P_ms","T_C_ms","T_O_ms",
        "time_sum_error_ms","qc_error_reason",
        "calculation_method","used_fixation_fallback",
    ]
    cols = [c for c in cols if c in problem_rows.columns]

    print("\nQC problem rows:")
    display(problem_rows[cols].head(200))
else:
    print("\nNo row-level QC problems found.")

quality_summary.to_csv(
    OUTPUT_DIR / "data_quality_summary.csv",
    index=False
)
problem_rows.to_csv(
    OUTPUT_DIR / "data_quality_problem_rows.csv",
    index=False
)


## 7. Condition × Image participant-count QC

In [ ]:

cell_qc = (
    master_all
    .groupby(["condition","image_id"], as_index=False)
    .agg(
        n_rows=("participant_id","size"),
        n_participants=("participant_id","nunique"),
        n_qc_pass=("row_qc_pass","sum"),
        n_qc_fail=("row_qc_pass", lambda s: int((~s).sum())),
    )
)

cell_qc["expected_n_participants"] = EXPECTED_PARTICIPANTS_PER_CELL
cell_qc["participant_count_ok"] = (
    cell_qc["n_participants"] == EXPECTED_PARTICIPANTS_PER_CELL
)
cell_qc["all_rows_qc_pass"] = (
    cell_qc["n_qc_fail"] == 0
)

def cell_error_reason(row):
    reasons = []
    if not row["participant_count_ok"]:
        reasons.append(
            f"participant count {row['n_participants']} "
            f"!= expected {EXPECTED_PARTICIPANTS_PER_CELL}"
        )
    if not row["all_rows_qc_pass"]:
        reasons.append(
            f"{row['n_qc_fail']} participant row(s) failed QC"
        )
    return "; ".join(reasons)

cell_qc["cell_qc_error_reason"] = cell_qc.apply(
    cell_error_reason, axis=1
)

display(cell_qc.sort_values(["image_id","condition"]))

cell_problems = cell_qc[
    cell_qc["cell_qc_error_reason"] != ""
].copy()

if len(cell_problems):
    print("\nCondition × Image cells with problems:")
    display(cell_problems)
else:
    print("\nAll loaded Condition × Image cells passed participant-count and row QC.")

participant_counts = (
    master_all
    .groupby(["image_id","condition"])["participant_id"]
    .nunique()
    .unstack("condition")
    .reindex(columns=CONDITIONS)
    .sort_index()
)

print("\nParticipant-count matrix:")
display(participant_counts)

cell_qc.to_csv(
    OUTPUT_DIR / "condition_image_QC.csv",
    index=False
)
cell_problems.to_csv(
    OUTPUT_DIR / "condition_image_QC_problems.csv",
    index=False
)
participant_counts.reset_index().to_csv(
    OUTPUT_DIR / "participant_counts_by_condition_image.csv",
    index=False
)


## 8. Create full and balanced datasets

In [ ]:

if any(len(present_sets[c]) == 0 for c in CONDITIONS):
    common_images = []
else:
    common_images = sorted(
        set.intersection(
            *(present_sets[c] for c in CONDITIONS)
        )
    )

master_balanced = master_all[
    master_all["image_id"].isin(common_images)
].copy()

master_all_qc_pass = master_all[
    master_all["row_qc_pass"]
].copy()

master_balanced_qc_pass = master_balanced[
    master_balanced["row_qc_pass"]
].copy()

print("Common images across A-E:", len(common_images))
print("Common image IDs:", common_images)
print()
print("ALL_AVAILABLE rows:", len(master_all))
print("BALANCED_COMMON_IMAGES rows:", len(master_balanced))
print("ALL_AVAILABLE QC-pass rows:", len(master_all_qc_pass))
print("BALANCED_COMMON_IMAGES QC-pass rows:", len(master_balanced_qc_pass))
print()
print(
    "Balanced Condition × Image cells:",
    len(common_images) * len(CONDITIONS)
)


## 9. Descriptive summaries

In [ ]:

SUMMARY_COLS = [
    "g_H","g_P","g_C","g_O",
    "semantic_attention",
    "semantic_entropy_bits",
    "semantic_entropy_normalized",
]

summary_all = (
    master_all
    .groupby("condition")[SUMMARY_COLS]
    .agg(["count","mean","std","median"])
)

summary_balanced = (
    master_balanced
    .groupby("condition")[SUMMARY_COLS]
    .agg(["count","mean","std","median"])
    if len(master_balanced)
    else pd.DataFrame()
)

print("ALL AVAILABLE")
display(summary_all)

print("\nBALANCED COMMON IMAGES")
display(summary_balanced)

summary_all.to_csv(
    OUTPUT_DIR / "condition_summary_ALL_AVAILABLE.csv"
)
summary_balanced.to_csv(
    OUTPUT_DIR / "condition_summary_BALANCED_COMMON_IMAGES.csv"
)


## 10. Save final datasets and diagnostic package

In [ ]:

all_path = OUTPUT_DIR / "master_semantic_attention_ALL_AVAILABLE.csv"
balanced_path = OUTPUT_DIR / "master_semantic_attention_BALANCED_COMMON_IMAGES.csv"
all_qc_path = OUTPUT_DIR / "master_semantic_attention_ALL_AVAILABLE_QC_PASS.csv"
balanced_qc_path = OUTPUT_DIR / "master_semantic_attention_BALANCED_COMMON_IMAGES_QC_PASS.csv"

master_all.to_csv(all_path, index=False)
master_balanced.to_csv(balanced_path, index=False)
master_all_qc_pass.to_csv(all_qc_path, index=False)
master_balanced_qc_pass.to_csv(balanced_qc_path, index=False)

print("Saved:")
print(" ", all_path)
print(" ", balanced_path)
print(" ", all_qc_path)
print(" ", balanced_qc_path)

print("\nDiagnostic reports:")
for name in [
    "coverage_matrix.csv",
    "missing_images_by_condition.csv",
    "schema_audit_all_files.csv",
    "schema_audit_failed_files.csv",
    "processing_report.csv",
    "processing_failed_files.csv",
    "data_quality_summary.csv",
    "data_quality_problem_rows.csv",
    "condition_image_QC.csv",
    "condition_image_QC_problems.csv",
    "participant_counts_by_condition_image.csv",
]:
    print(" ", OUTPUT_DIR / name)

print("\nRecommended direct A-E comparison dataset:")
print(" ", balanced_path)



## Expected result for your current coverage

Based on the missing files you already identified:

```text
D,5
E,18
```

the notebook should report:

- **98 / 100 available Condition × Image cells**
- **18 images common to all five conditions**
- **90 balanced Condition × Image cells**

If any CSV cannot be loaded, the notebook will show:

- the exact file;
- error type;
- error message;
- missing columns;
- available columns;
- schema mapping used.

So you will know **why** it failed instead of only seeing a generic `RuntimeError`.
